# Frozen exact shard: Colab CPU worker

Upload `colab_frozen_worker_input.tar.zst` and its `.sha256` file from the laptop to `MyDrive/amazon_exact_transfer/`. Select a **High-RAM CPU runtime** if available. Run cells in order. The first run benchmarks one real full-target production shard (index 0). It uses the frozen K100 routes and cached 50/50 model. Do not submit any partial file.

Drive checkpoints are written after each complete retrieval, augmentation, and inference phase. Reconnect and rerun cells to resume. If Colab terminates during a phase, that phase restarts.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, shutil, subprocess, sys, hashlib, json, time
DRIVE = Path('/content/drive/MyDrive/amazon_exact_transfer')
BUNDLE = DRIVE/'colab_frozen_worker_input.tar.zst'
CHECK = DRIVE/'colab_frozen_worker_input.tar.zst.sha256'
assert BUNDLE.is_file() and CHECK.is_file(), 'Upload the bundle and .sha256 file to MyDrive/amazon_exact_transfer first'
print('Drive bundle found:', BUNDLE, f'{BUNDLE.stat().st_size/2**30:.2f} GiB')


In [ ]:
# Install only runtime packages and the zstd decompressor; no external business data is fetched.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'zstandard'], check=True)
subprocess.run(['bash','-lc','command -v zstd >/dev/null || (apt-get update -qq && apt-get install -y -qq zstd)'], check=True)
LOCAL = Path('/content/amazon_exact')
LOCAL.mkdir(exist_ok=True)
local_bundle = Path('/content/colab_frozen_worker_input.tar.zst')
if not local_bundle.exists() or local_bundle.stat().st_size != BUNDLE.stat().st_size:
    shutil.copy2(BUNDLE, local_bundle)
def sha(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''): h.update(block)
    return h.hexdigest()
expected=CHECK.read_text().split()[0]
assert sha(local_bundle)==expected, 'Bundle SHA-256 mismatch'
if not (LOCAL/'research_runs/submission_exact_shards/shards.json').exists():
    subprocess.run(['tar','-I','zstd','-xf',str(local_bundle),'-C',str(LOCAL)],check=True)
print('Bundle extracted and checksum verified')


In [ ]:
REQ = LOCAL/'code/business_entity_resolution/requirements.txt'
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REQ)],check=True)
subprocess.run(['bash',str(LOCAL/'code/business_entity_resolution/bootstrap_distributed.sh')],cwd=LOCAL,check=True,env={**os.environ,'PYTHON_BIN':sys.executable})
import psutil
print('Python:',sys.version.split()[0], 'RAM GiB:',round(psutil.virtual_memory().total/2**30,2), 'CPU:',psutil.cpu_count())
assert psutil.virtual_memory().total >= 8*2**30, 'Use a High-RAM runtime; full-shard safety is untested below 8 GiB'


## One real production shard

This cell runs shard 0 over every test S2/S3 target and writes completed phase archives plus SHA-256 sidecars to Drive. Keep the runtime connected. Re-running after interruption restores the latest completed phase. Do not start another shard until this one completes and its timing has been reviewed.


In [ ]:
SHARD_INDEX = 0
WORKERS = 2  # Reduce concurrent processes on standard Colab RAM; retrieval semantics stay frozen.
REMOTE = DRIVE/'completed_shards'
REMOTE.mkdir(exist_ok=True)
command=[sys.executable,str(LOCAL/'code/business_entity_resolution/15_colab_checkpoint.py'),
         '--shard-index',str(SHARD_INDEX),'--num-shards','174',
         '--run-dir',str(LOCAL/'research_runs/submission_exact_shards'),
         '--drive-dir',str(REMOTE),'--workers',str(WORKERS),'--target-chunk','50000']
log_path=REMOTE/f'{SHARD_INDEX:05d}_worker.log'
with log_path.open('a',buffering=1) as log:
    log.write('START '+time.strftime('%Y-%m-%d %H:%M:%S %Z')+'\n')
    proc=subprocess.Popen(command,cwd=LOCAL,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in proc.stdout:
        print(line,end='')
        log.write(line)
    rc=proc.wait()
    log.write(f'EXIT {rc}\n')
assert rc==0, f'Shard failed (exit {rc}); inspect {log_path} and rerun this cell after addressing the error'


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
import math
manifest=json.loads((LOCAL/'research_runs/submission_exact_shards/pred'/f'{SHARD_INDEX:05d}'/'completion_manifest.json').read_text())
assert manifest['completion'] is True and manifest['target_count']==9969589
seconds=manifest['runtime_seconds']
print('REAL SHARD:',json.dumps({'runtime_hours':None if seconds is None else round(seconds/3600,3),
 'peak_ram_gib':round(manifest['sampled_peak_rss_bytes']/2**30,3),
 'phase_cpu_seconds':{p:v.get('sampled_cpu_seconds') for p,v in manifest['phases'].items()},
 'phase_io_bytes':{p:(v.get('sampled_read_bytes'),v.get('sampled_write_bytes')) for p,v in manifest['phases'].items()},
 'base_target_rows_per_second':None if not manifest['phases']['retrieve']['seconds'] else round(9969589/manifest['phases']['retrieve']['seconds'],1),
 'drive_complete_archive':str(REMOTE/f'{SHARD_INDEX:05d}_infer.tar.gz')},indent=2))
# Assumption: the 11:59 PM IST deadline is today; adjust if the portal shows a different date.
deadline=datetime.now(ZoneInfo('Asia/Kolkata')).replace(hour=23,minute=59,second=0,microsecond=0)
remaining=(deadline-datetime.now(ZoneInfo('Asia/Kolkata'))).total_seconds()
print('Remaining hours to deadline:',round(remaining/3600,2))
if seconds is None:
 print('Full runtime unavailable: one or more phases resumed without saved timing.')
else:
 for sessions in (1,2,4,8,16,32):
  projected=math.ceil(174/sessions)*seconds*1.25/3600
  print(f'{sessions:2d} Colab sessions: ~{projected:.2f} h incl 25% transfer/overhead; before deadline={projected*3600<=remaining}')
 viable=[n for n in range(1,175) if math.ceil(174/n)*seconds*1.25<=remaining]
 print('Minimum modeled concurrent sessions:',viable[0] if viable else 'none at measured rate')


## Transfer back to the laptop

Download the three `00000_{retrieve,augment,infer}.tar.gz` files and their matching `.sha256` sidecars from `MyDrive/amazon_exact_transfer/completed_shards/`. The laptop imports and verifies them with:

```bash
python code/business_entity_resolution/16_import_colab_checkpoints.py \
  --archive-dir /path/to/downloaded/completed_shards \
  --shard-index 0
```

Only after **all 174** completed manifests have been imported, use the merge, official validator, and package commands in `SUBMISSION_RUNBOOK.md`.
